
##### 02 - Silver Layer Transformation

The Silver layer contains cleansed, deduplicated, and typed data.

**Transformations applied:**
- Data type casting & standardization
- Null handling & default values
- Deduplication using window functions
- Referential integrity checks
- Business rule validation


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DateType, TimestampType, DoubleType, IntegerType

##### 1. Silver Customers

In [0]:
df_customers_bronze = spark.table("ecommerce_bronze.customers")

print(f"Bronze customer count: {df_customers_bronze.count()}")
print(f"Null emails: {df_customers_bronze.filter(F.col('email').isNull()).count()}")
print(f"Null phones: {df_customers_bronze.filter(F.col('phone').isNull()).count()}")

dedup_window = Window.partitionBy("customer_id").orderBy(F.col("_ingestion_timestamp").desc())

df_customers_silver = (
    df_customers_bronze
    .withColumn("_row_num", F.row_number().over(dedup_window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num", "_source_file", "_batch_id", "_ingestion_timestamp")

    .withColumn("signup_date", F.to_date(F.col("signup_date"), "yyyy-MM-dd"))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("first_name", F.initcap(F.trim(F.col("first_name"))))
    .withColumn("last_name", F.initcap(F.trim(F.col("last_name"))))
    .withColumn("full_name", F.concat_ws(" ", F.col("first_name"), F.col("last_name")))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("state", F.initcap(F.trim(F.col("state"))))

    .withColumn("email", F.when(F.col("email").rlike(r"^[\w\.\-]+@[\w\.\-]+\.\w+$"), F.col("email"))
                .otherwise(None))

    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("age").between(13, 120) | F.col("age").isNull())

    .withColumn("_silver_timestamp", F.current_timestamp())
)

(
    df_customers_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_silver.customers")
)

print(f"Silver customers: {df_customers_silver.count()}")
df_customers_silver.select("customer_id", "full_name", "email", "city", "signup_date").show(5, truncate=False)


Bronze customer count: 10000
Null emails: 329
Null phones: 244
Silver customers: 10000
+-----------+------------+--------------------------+---------+-----------+
|customer_id|full_name   |email                     |city     |signup_date|
+-----------+------------+--------------------------+---------+-----------+
|CUST-000001|Rohan Singh |rohan.singh760@outlook.com|Mumbai   |2024-02-22 |
|CUST-000002|Myra Verma  |myra.verma96@yahoo.com    |Mumbai   |2024-01-14 |
|CUST-000003|Myra Nair   |myra.nair604@outlook.com  |Ahmedabad|2024-12-23 |
|CUST-000004|Vikram Mehta|vikram.mehta95@hotmail.com|Delhi    |2024-11-05 |
|CUST-000005|Vihaan Rao  |vihaan.rao566@outlook.com |Delhi    |2024-07-04 |
+-----------+------------+--------------------------+---------+-----------+
only showing top 5 rows


##### 2. Silver Products

In [0]:
df_products_bronze = spark.table("ecommerce_bronze.products")

dedup_window_prod = Window.partitionBy("product_id").orderBy(F.col("_ingestion_timestamp").desc())

df_products_silver = (
    df_products_bronze
    .withColumn("_row_num", F.row_number().over(dedup_window_prod))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num", "_source_file", "_batch_id", "_ingestion_timestamp")

    .withColumn("price", F.col("price").cast(DoubleType()))
    .withColumn("rating", F.round(F.col("rating").cast(DoubleType()), 1))
    .withColumn("stock_quantity", F.col("stock_quantity").cast(IntegerType()))
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.trim(F.col("category")))

    .filter(F.col("product_id").isNotNull())
    .filter(F.col("price") > 0)
    .withColumn("rating", F.when(F.col("rating").between(0, 5), F.col("rating")).otherwise(None))

    .withColumn("price_tier",
                F.when(F.col("price") < 500, "Budget")
                .when(F.col("price") < 2000, "Mid-Range")
                .when(F.col("price") < 10000, "Premium")
                .otherwise("Luxury"))

    .withColumn("_silver_timestamp", F.current_timestamp())
)

(
    df_products_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_silver.products")
)

print(f"Silver products: {df_products_silver.count()}")
df_products_silver.select("product_id", "product_name", "category", "price", "price_tier").show(5, truncate=False)


Silver products: 200
+----------+------------+-----------+--------+----------+
|product_id|product_name|category   |price   |price_tier|
+----------+------------+-----------+--------+----------+
|PROD-0001 |Smartphone  |Electronics|40729.29|Luxury    |
|PROD-0002 |Laptop      |Electronics|58038.4 |Luxury    |
|PROD-0003 |Headphones  |Electronics|3398.81 |Premium   |
|PROD-0004 |Tablet      |Electronics|57171.71|Luxury    |
|PROD-0005 |Smartwatch  |Electronics|24243.77|Luxury    |
+----------+------------+-----------+--------+----------+
only showing top 5 rows


##### 3. Silver Orders

In [0]:
df_orders_bronze = spark.table("ecommerce_bronze.orders")

print(f"Bronze order count (before dedup): {df_orders_bronze.count()}")
print(f"Distinct order IDs: {df_orders_bronze.select('order_id').distinct().count()}")
print(f"Duplicate records: {df_orders_bronze.count() - df_orders_bronze.select('order_id').distinct().count()}")

dedup_window_ord = Window.partitionBy("order_id").orderBy(F.col("_ingestion_timestamp").desc())

valid_customer_ids = df_customers_silver.select("customer_id")

df_orders_silver = (
    df_orders_bronze
    .withColumn("_row_num", F.row_number().over(dedup_window_ord))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num", "_source_file", "_batch_id", "_ingestion_timestamp")

    .withColumn("order_date", F.to_date(F.col("order_date"), "yyyy-MM-dd"))
    .withColumn("subtotal", F.col("subtotal").cast(DoubleType()))
    .withColumn("shipping_fee", F.col("shipping_fee").cast(DoubleType()))
    .withColumn("total_amount", F.col("total_amount").cast(DoubleType()))
    .withColumn("status", F.lower(F.trim(F.col("status"))))
    .withColumn("payment_method", F.trim(F.col("payment_method")))

    .filter(F.col("order_id").isNotNull())
    .filter(F.col("total_amount") >= 0)
    .filter(F.col("order_date").isNotNull())

    .join(valid_customer_ids, "customer_id", "inner")

    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
    .withColumn("order_day_of_week", F.dayofweek("order_date"))

    .withColumn("_silver_timestamp", F.current_timestamp())
)

(
    df_orders_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_silver.orders")
)

print(f"Silver orders: {df_orders_silver.count()}")


Bronze order count (before dedup): 50255
Distinct order IDs: 50000
Duplicate records: 255
Silver orders: 50000


##### 4. Silver Order Items

In [0]:
df_items_bronze = spark.table("ecommerce_bronze.order_items")

valid_order_ids = df_orders_silver.select("order_id")
valid_product_ids = df_products_silver.select("product_id")

dedup_window_items = Window.partitionBy("item_id").orderBy(F.col("_ingestion_timestamp").desc())

df_items_silver = (
    df_items_bronze
    .withColumn("_row_num", F.row_number().over(dedup_window_items))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num", "_source_file", "_batch_id", "_ingestion_timestamp")

    .withColumn("quantity", F.col("quantity").cast(IntegerType()))
    .withColumn("unit_price", F.col("unit_price").cast(DoubleType()))
    .withColumn("discount_pct", F.col("discount_pct").cast(DoubleType()))
    .withColumn("line_total", F.col("line_total").cast(DoubleType()))

    .filter(F.col("item_id").isNotNull())
    .filter(F.col("quantity") > 0)
    .filter(F.col("unit_price") > 0)

    .join(valid_order_ids, "order_id", "inner")
    .join(valid_product_ids, "product_id", "inner")

    .withColumn("_silver_timestamp", F.current_timestamp())
)

(
    df_items_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_silver.order_items")
)

print(f"Silver order_items: {df_items_silver.count()}")

Silver order_items: 105115


##### 5. Silver Layer Summary

In [0]:
%sql
SELECT 'customers' AS table_name,
       (SELECT COUNT(*) FROM ecommerce_bronze.customers) AS bronze_count,
       (SELECT COUNT(*) FROM ecommerce_silver.customers) AS silver_count
UNION ALL
SELECT 'products',
       (SELECT COUNT(*) FROM ecommerce_bronze.products),
       (SELECT COUNT(*) FROM ecommerce_silver.products)
UNION ALL
SELECT 'orders',
       (SELECT COUNT(*) FROM ecommerce_bronze.orders),
       (SELECT COUNT(*) FROM ecommerce_silver.orders)
UNION ALL
SELECT 'order_items',
       (SELECT COUNT(*) FROM ecommerce_bronze.order_items),
       (SELECT COUNT(*) FROM ecommerce_silver.order_items)
ORDER BY table_name


table_name,bronze_count,silver_count
customers,10000,10000
order_items,105115,105115
orders,50255,50000
products,200,200


In [0]:
print("Silver transformation complete! Proceed to notebook 03_gold_aggregation.")

Silver transformation complete! Proceed to notebook 03_gold_aggregation.
